In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Sat Jun 27 15:48:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
%cd /content

!git clone https://github.com/graphdeco-inria/gaussian-splatting.git

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 16.12 MiB/s, done.
Resolving deltas: 100% (595/595), done.


In [4]:
%cd /content/gaussian-splatting

/content/gaussian-splatting


In [5]:
import sys

print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [6]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA: 12.8
GPU: NVIDIA A100-SXM4-40GB


In [7]:
!git branch
!git log --oneline -1

* main
54c035f (HEAD -> main, origin/main, origin/HEAD) fix submodule name in environment.yml


In [8]:
!ls

arguments   environment.yml    LICENSE.md    README.md	 scene	       train.py
assets	    full_eval.py       lpipsPyTorch  render.py	 SIBR_viewers  utils
convert.py  gaussian_renderer  metrics.py    results.md  submodules


In [9]:
!ls submodules

diff-gaussian-rasterization  fused-ssim  simple-knn


In [10]:
!pip install plyfile tqdm lpips opencv-python imageio scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.3 MB/s eta 0:00:00


In [11]:
!cat environment.yml

name: gaussian_splatting
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - cudatoolkit=11.6
  - plyfile
  - python=3.7.13
  - pip=22.3.1
  - pytorch=1.12.1
  - torchaudio=0.12.1
  - torchvision=0.13.1
  - tqdm
  - pip:
    - submodules/diff-gaussian-rasterization
    - submodules/simple-knn
    - submodules/fused-ssim
    - opencv-python
    - joblib


In [13]:
!find submodules -maxdepth 2 \( -name "setup.py" -o -name "pyproject.toml" \)

In [12]:
!find . -maxdepth 2 -name "setup.py"

In [14]:
!ls -lah submodules/diff-gaussian-rasterization

total 8.0K
drwxr-xr-x 2 root root 4.0K Jun 27 15:48 .
drwxr-xr-x 5 root root 4.0K Jun 27 15:48 ..


In [15]:
!ls -lah submodules/simple-knn

total 8.0K
drwxr-xr-x 2 root root 4.0K Jun 27 15:48 .
drwxr-xr-x 5 root root 4.0K Jun 27 15:48 ..


In [16]:
!ls -lah submodules/fused-ssim

total 8.0K
drwxr-xr-x 2 root root 4.0K Jun 27 15:48 .
drwxr-xr-x 5 root root 4.0K Jun 27 15:48 ..


In [17]:
!git submodule update --init --recursive

Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
Cloning into '/content/gaussian-splatting/submodules/diff-gaussian-rasterization'...
Cloning into '/content/gaussian-splatting/submodules/fused-ssim'...
Cloning into '/content/gaussian-splatting/submodules/simple-knn'...
Submodule path 'SIBR_viewers': checked out 'd8856f60c5384cc1975439193bb627d77d917d77'
Submodule path 'submodules/diff-gaussian-rasterization': checked out '9c5c2028f6fbee2be239bc4c942

In [18]:
!ls submodules/diff-gaussian-rasterization

CMakeLists.txt		     LICENSE.md		  setup.py
cuda_rasterizer		     rasterize_points.cu  third_party
diff_gaussian_rasterization  rasterize_points.h
ext.cpp			     README.md


In [19]:
%cd /content/gaussian-splatting

!pip install -e submodules/diff-gaussian-rasterization

/content/gaussian-splatting
Obtaining file:///content/gaussian-splatting/submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Running setup.py develop for diff_gaussian_rasterization


In [20]:
%cd /content/gaussian-splatting

!pip install -e submodules/simple-knn

/content/gaussian-splatting
Obtaining file:///content/gaussian-splatting/submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Running setup.py develop for simple_knn


In [21]:
%cd /content/gaussian-splatting

!pip install -e submodules/fused-ssim

/content/gaussian-splatting
Obtaining file:///content/gaussian-splatting/submodules/fused-ssim
  Preparing metadata (setup.py) ... done
  Running setup.py develop for fused_ssim


In [23]:
!pip show diff-gaussian-rasterization

Name: diff_gaussian_rasterization
Version: 0.0.0
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: /content/gaussian-splatting/submodules/diff-gaussian-rasterization
Editable project location: /content/gaussian-splatting/submodules/diff-gaussian-rasterization
Requires: 
Required-by: 


In [24]:
!pip show simple-knn

Name: simple_knn
Version: 0.0.0
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: /content/gaussian-splatting/submodules/simple-knn
Editable project location: /content/gaussian-splatting/submodules/simple-knn
Requires: 
Required-by: 


In [25]:
!pip show fused-ssim

Name: fused_ssim
Version: 0.0.0
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: /content/gaussian-splatting/submodules/fused-ssim
Editable project location: /content/gaussian-splatting/submodules/fused-ssim
Requires: 
Required-by: 


In [26]:
!pip list | grep -E "gaussian|knn|ssim"

diff_gaussian_rasterization              0.0.0              /content/gaussian-splatting/submodules/diff-gaussian-rasterization
fused_ssim                               0.0.0              /content/gaussian-splatting/submodules/fused-ssim
simple_knn                               0.0.0              /content/gaussian-splatting/submodules/simple-knn


In [27]:
!python train.py --help

2026-06-27 16:04:29.635878: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 16:04:29.707225: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
usage: train.py [-h] [--sh_degree SH_DEGREE] [--source_path SOURCE_PATH]
                [--model_path MODEL_PATH] [--images IMAGES] [--depths DEPTHS]
                [--resolution RESOLUTION] [--white_background]
                [--train_test_exp] [--data_device DATA_DEVICE] [--eval]
                [--iterations ITERATIONS]
                [--position_lr_init POSITION_L

In [28]:
from google.colab import files
uploaded = files.upload()

In [30]:
!ls /content

ArturAzevedoBusto.zip  dataset	drive  gaussian-splatting  sample_data


In [31]:
!rm -rf /content/dataset
!mkdir -p /content/dataset

!unzip -q /content/ArturAzevedoBusto.zip -d /content/dataset

In [32]:
!find /content/dataset -maxdepth 3

/content/dataset
/content/dataset/ArturAzevedoBusto
/content/dataset/ArturAzevedoBusto/IMG_4376.jpg
/content/dataset/ArturAzevedoBusto/IMG_4357.jpg
/content/dataset/ArturAzevedoBusto/IMG_4354.jpg
/content/dataset/ArturAzevedoBusto/IMG_4375.jpg
/content/dataset/ArturAzevedoBusto/IMG_4366.jpg
/content/dataset/ArturAzevedoBusto/IMG_4386.jpg
/content/dataset/ArturAzevedoBusto/IMG_4389.jpg
/content/dataset/ArturAzevedoBusto/IMG_4388.jpg
/content/dataset/ArturAzevedoBusto/IMG_4374.jpg
/content/dataset/ArturAzevedoBusto/IMG_4362.jpg
/content/dataset/ArturAzevedoBusto/IMG_4365.jpg
/content/dataset/ArturAzevedoBusto/IMG_4380.jpg
/content/dataset/ArturAzevedoBusto/IMG_4371.jpg
/content/dataset/ArturAzevedoBusto/IMG_4373.jpg
/content/dataset/ArturAzevedoBusto/IMG_4379.jpg
/content/dataset/ArturAzevedoBusto/IMG_4342.jpg
/content/dataset/ArturAzevedoBusto/IMG_4382.jpg
/content/dataset/ArturAzevedoBusto/IMG_4368.jpg
/content/dataset/ArturAzevedoBusto/IMG_4343.jpg
/content/dataset/ArturAzevedoBusto/I

In [34]:
!apt-get update
!DEBIAN_FRONTEND=noninteractive apt-get install -y colmap

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

In [35]:
!colmap -h

COLMAP 3.7 -- Structure-from-Motion and Multi-View Stereo
              (Commit Unknown on Unknown without CUDA)

Usage:
  colmap [command] [options]

Documentation:
  https://colmap.github.io/

Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...

Available commands:
  help
  gui
  automatic_reconstructor
  bundle_adjuster
  color_extractor
  database_cleaner
  database_creator
  database_merger
  delaunay_mesher
  exhaustive_matcher
  feature_extractor
  feature_importer
  hierarchical_mapper
  image_deleter
  image_filterer
  image_rectifier
  image_registrator
  image_undistorter
  image_undistorter_standalone
  mapper

In [36]:
!mkdir -p /content/dataset/ArturAzevedoBusto/distorted/sparse

In [38]:
!which colmap

/usr/bin/colmap


In [39]:
!colmap feature_extractor -h

COLMAP 3.7 (Commit Unknown on Unknown without CUDA)

Options can either be specified via command-line or by defining
them in a .ini project file passed to `--project_path`.

  -h [ --help ] 
  --random_seed arg (=0)
  --log_to_stderr arg (=0)
  --log_level arg (=2)
  --project_path arg
  --database_path arg
  --image_path arg
  --camera_mode arg (=-1)
  --image_list_path arg
  --descriptor_normalization arg (=l1_root)
                                        {'l1_root', 'l2'}
  --ImageReader.mask_path arg
  --ImageReader.camera_model arg (=SIMPLE_RADIAL)
  --ImageReader.single_camera arg (=0)
  --ImageReader.single_camera_per_folder arg (=0)
  --ImageReader.single_camera_per_image arg (=0)
  --ImageReader.existing_camera_id arg (=-1)
  --ImageReader.camera_params arg
  --ImageReader.default_focal_length_factor arg (=1.2)
  --ImageReader.camera_mask_path arg
  --SiftExtraction.num_threads arg (=-1)
  --SiftExtraction.use_gpu arg (=1)
  --SiftExtraction.gpu_index arg (=-1)
  --SiftExtract

In [40]:
!colmap feature_extractor \
    --database_path /content/dataset/ArturAzevedoBusto/distorted/database.db \
    --image_path /content/dataset/ArturAzevedoBusto \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/51]
  Name:            IMG_4351.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        12250
Processed file [2/51]
  Name:            IMG_4350.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        12872
Processed file [3/51]
  Name:            IMG_4343.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        12706
Processed file [4/51]
  Name:            IMG_4347.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        13774
Processed file [5/51]
  Name:            IMG_4348.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        13877
Processed file [6/51]
  Name:            IMG_4352.jpg
  Dimensions:      3024 x 4032
  Camera:

In [41]:
!mkdir -p /content/dataset/ArturAzevedoBusto/images

!find /content/dataset/ArturAzevedoBusto \
-maxdepth 1 \
-name "*.jpg" \
-exec mv {} /content/dataset/ArturAzevedoBusto/images/ \;

In [42]:
!ls /content/dataset/ArturAzevedoBusto/images | wc -l

48


In [43]:
!find /content/dataset/ArturAzevedoBusto -maxdepth 1 -name "*.jpg" | wc -l

0


In [44]:
!find /content/dataset/ArturAzevedoBusto/images -name "*.jpg" | wc -l

48


In [45]:
!ls -lah /content/dataset/ArturAzevedoBusto

total 16K
drwxrwxrwx 4 root root 4.0K Jun 27 16:25 .
drwxr-xr-x 3 root root 4.0K Jun 27 16:16 ..
drwxr-xr-x 3 root root 4.0K Jun 27 16:25 distorted
drwxr-xr-x 2 root root 4.0K Jun 27 16:25 images


In [46]:
!find /content/dataset/ArturAzevedoBusto/images -type f | sort

/content/dataset/ArturAzevedoBusto/images/IMG_4342.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4343.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4344.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4345.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4346.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4347.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4348.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4349.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4350.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4351.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4352.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4353.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4354.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4355.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4356.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4357.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4358.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4359.jpg
/content/d

In [47]:
!find /content/dataset/ArturAzevedoBusto/images -type f | wc -l

48


In [48]:
!colmap exhaustive_matcher \
    --database_path /content/dataset/ArturAzevedoBusto/distorted/database.db \
    --SiftMatching.use_gpu 0


Exhaustive feature matching

Matching block [1/1, 1/1] in 270.128s
Elapsed time: 4.502 [minutes]


In [49]:
!mkdir -p /content/dataset/ArturAzevedoBusto/distorted/sparse

!colmap mapper \
    --database_path /content/dataset/ArturAzevedoBusto/distorted/database.db \
    --image_path /content/dataset/ArturAzevedoBusto/images \
    --output_path /content/dataset/ArturAzevedoBusto/distorted/sparse


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 452 in 0.009s
Loading images... 48 in 0.035s (connected 48)
Building correspondence graph... in 0.099s (ignored 0)

Elapsed time: 0.002 [minutes]


Finding good initial image pair


Initializing with image pair #8 and #5


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  2.621308e+03    0.00e+00    7.99e+05   0.00e+00   0.00e+00  1.00e+04        0    4.62e-03    1.20e-02
   1  1.228487e+03    1.39e+03    7.00e+05   8.17e+01   8.34e-01  1.42e+04        1    8.07e-03    2.01e-02
   2  7.999854e+02    4.29e+02    4.19e+05   7.19e+01   7.98e-01  1.80e+04        1    7.70e-03    2.79e-02
   3  6.053488e+02    1.95e+02    1.41e+05   5.38e+01   9.12e-01  4.08e+04        1    7.60e-03    3.55e-02
   4  5.483855e+02    5.70e+01    4.93e+04   4.81e+01   8.83e-01  7.40e+04        1    7.60e-03    4.31e-02
   5  5.333017e+02    1.51e+01    

In [51]:
!ls -R /content/dataset/ArturAzevedoBusto/distorted/sparse

/content/dataset/ArturAzevedoBusto/distorted/sparse:
0

/content/dataset/ArturAzevedoBusto/distorted/sparse/0:
cameras.bin  images.bin  points3D.bin  project.ini


In [53]:
!mkdir -p /content/dataset/ArturAzevedoBusto

!colmap image_undistorter \
    --image_path /content/dataset/ArturAzevedoBusto/images \
    --input_path /content/dataset/ArturAzevedoBusto/distorted/sparse/0 \
    --output_path /content/dataset/ArturAzevedoBusto \
    --output_type COLMAP


Reading reconstruction

 => Reconstruction with 48 images and 38815 points

Image undistortion

Undistorting image [1/48]
Undistorting image [2/48]
Undistorting image [3/48]
Undistorting image [4/48]
Undistorting image [5/48]
Undistorting image [6/48]
Undistorting image [7/48]
Undistorting image [8/48]
Undistorting image [9/48]
Undistorting image [10/48]
Undistorting image [11/48]
Undistorting image [12/48]
Undistorting image [13/48]
Undistorting image [14/48]
Undistorting image [15/48]
Undistorting image [16/48]
Undistorting image [17/48]
Undistorting image [18/48]
Undistorting image [19/48]
Undistorting image [20/48]
Undistorting image [21/48]
Undistorting image [22/48]
Undistorting image [23/48]
Undistorting image [24/48]
Undistorting image [25/48]
Undistorting image [26/48]
Undistorting image [27/48]
Undistorting image [28/48]
Undistorting image [29/48]
Undistorting image [30/48]
Undistorting image [31/48]
Undistorting image [32/48]
Undistorting image [33/48]
Undistorting image [3

In [54]:
!find /content/dataset/ArturAzevedoBusto -maxdepth 2

/content/dataset/ArturAzevedoBusto
/content/dataset/ArturAzevedoBusto/run-colmap-photometric.sh
/content/dataset/ArturAzevedoBusto/run-colmap-geometric.sh
/content/dataset/ArturAzevedoBusto/stereo
/content/dataset/ArturAzevedoBusto/stereo/consistency_graphs
/content/dataset/ArturAzevedoBusto/stereo/fusion.cfg
/content/dataset/ArturAzevedoBusto/stereo/patch-match.cfg
/content/dataset/ArturAzevedoBusto/stereo/normal_maps
/content/dataset/ArturAzevedoBusto/stereo/depth_maps
/content/dataset/ArturAzevedoBusto/distorted
/content/dataset/ArturAzevedoBusto/distorted/database.db
/content/dataset/ArturAzevedoBusto/distorted/sparse
/content/dataset/ArturAzevedoBusto/images
/content/dataset/ArturAzevedoBusto/images/IMG_4376.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4357.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4354.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4375.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4366.jpg
/content/dataset/ArturAzevedoBusto/images/IMG_4386.jp

In [55]:
%cd /content/gaussian-splatting

!mkdir -p /content/output_3dgs

/content/gaussian-splatting


In [57]:
import os
import shutil

os.makedirs("/content/dataset/ArturAzevedoBusto/sparse/0", exist_ok=True)

for f in ["cameras.bin", "images.bin", "points3D.bin"]:
    src = f"/content/dataset/ArturAzevedoBusto/sparse/{f}"
    dst = f"/content/dataset/ArturAzevedoBusto/sparse/0/{f}"
    shutil.copy(src, dst)

print("Estrutura corrigida!")

Estrutura corrigida!


In [58]:
!ls /content/dataset/ArturAzevedoBusto/sparse/0

cameras.bin  images.bin  points3D.bin


In [60]:
%cd /content/gaussian-splatting

!rm -rf /content/output_3dgs

!python train.py \
    -s /content/dataset/ArturAzevedoBusto \
    -m /content/output_3dgs \
    --eval \
    --iterations 7000

/content/gaussian-splatting
2026-06-27 16:44:08.397562: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 16:44:08.468451: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/output_3dgs
Output folder: /content/output_3dgs [27/06 16:44:12]
------------LLFF HOLD------------- [27/06 16:44:12]
Reading camera 48/48 [27/06 16:44:12]
Loading Training Cameras [27/06 16:44:12]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly

In [2]:
!mkdir -p /content/dataset/Leao

!unzip -q /content/leao.zip \
-d /content/dataset/Leao

In [3]:
!find /content/dataset/Leao -maxdepth 2

/content/dataset/Leao
/content/dataset/Leao/leao
/content/dataset/Leao/leao/IMG_4204.jpg
/content/dataset/Leao/leao/IMG_4212.jpg
/content/dataset/Leao/leao/IMG_4195.jpg
/content/dataset/Leao/leao/IMG_4201.jpg
/content/dataset/Leao/leao/IMG_4185.jpg
/content/dataset/Leao/leao/IMG_4224.jpg
/content/dataset/Leao/leao/IMG_4192.jpg
/content/dataset/Leao/leao/IMG_4232.jpg
/content/dataset/Leao/leao/IMG_4219.jpg
/content/dataset/Leao/leao/IMG_4217.jpg
/content/dataset/Leao/leao/IMG_4188.jpg
/content/dataset/Leao/leao/IMG_4197.jpg
/content/dataset/Leao/leao/IMG_4200.jpg
/content/dataset/Leao/leao/IMG_4229.jpg
/content/dataset/Leao/leao/IMG_4191.jpg
/content/dataset/Leao/leao/IMG_4220.jpg
/content/dataset/Leao/leao/IMG_4222.jpg
/content/dataset/Leao/leao/IMG_4214.jpg
/content/dataset/Leao/leao/IMG_4216.jpg
/content/dataset/Leao/leao/IMG_4206.jpg
/content/dataset/Leao/leao/IMG_4231.jpg
/content/dataset/Leao/leao/IMG_4215.jpg
/content/dataset/Leao/leao/IMG_4213.jpg
/content/dataset/Leao/leao/IMG_

In [4]:
import os
import shutil

base = "/content/dataset/Leao"

os.makedirs(f"{base}/images", exist_ok=True)

for arq in os.listdir(f"{base}/leao"):
    if arq.lower().endswith((".jpg", ".jpeg", ".png")):
        shutil.copy(f"{base}/leao/{arq}", f"{base}/images/{arq}")

print("Imagens copiadas!")
print("Quantidade:", len(os.listdir(f"{base}/images")))

Imagens copiadas!
Quantidade: 49


In [5]:
import os

os.makedirs("/content/dataset/Leao/distorted", exist_ok=True)

In [6]:
!colmap feature_extractor \
    --database_path /content/dataset/Leao/distorted/database.db \
    --image_path /content/dataset/Leao/images \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/49]
  Name:            IMG_4185.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        13137
Processed file [2/49]
  Name:            IMG_4187.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        13126
Processed file [3/49]
  Name:            IMG_4184.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        13191
Processed file [4/49]
  Name:            IMG_4192.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        14080
Processed file [5/49]
  Name:            IMG_4189.jpg
  Dimensions:      3024 x 4032
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    4838.40px
  Features:        14017
Processed file [6/49]
  Name:            IMG_4191.jpg
  Dimensions:      3024 x 4032
  Camera: